# 02 - LLaMEA Evolutionary Search Pipeline

This notebook is the main entry point for running LLaMEA evolution experiments. It supports:
1. **Single-Problem Evolution**: Quick testing and optimization of an algorithm for a single target BBOB problem.
2. **Multi-Problem Batch Evolution**: Looping over multiple BBOB problem IDs sequentially with automatic database storage.

In [ ]:
# Setup paths and imports
import os
import sys
from functools import partial
from pathlib import Path

# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Enable automatic reloading of modified modules
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

from core import LLaMEASession, dispatch, EvolutionJob
from core.problems.bbob import BBOBProblem
from core.llamea.prompts import TASK_PROMPT_NOISY, TASK_PROMPT_CLEAN, FORMAT_PROMPT
from infra.llm.client import get_llm_client
from infra.storage import initialize_storage

print("LLaMEA Evolution Pipeline initialized with Orchestrator.")

## 1. LLM Provider Connection
Initialize and test connection to the local LLM server configured in your `.env` file.

In [ ]:
env_path = root_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

llm_provider = os.getenv("LLM_PROVIDER", "local")

try:
    llm = get_llm_client(llm_provider)
    print("=================================================================")
    print(f"Initialized Provider: {llm_provider}")
    print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
    print("=================================================================")
except Exception as e:
    print(f"Provider setup notice: {e}")
    llm = None

## 2. Section 1: Single-Problem Evolution
Run LLaMEA optimization algorithm evolution for a single target BBOB problem configuration and persist the results.

In [ ]:
# Configuration for single run
problem_id = 1
dim = 3
instance_id = 1
noise_std = 0.05
iterations = 5
budget = 1000
mode = "noisy" if noise_std > 0.0 else "clean"

problem = BBOBProblem(problem_id=problem_id, dim=dim, instance_id=instance_id)

print(f"Target Problem: BBOB {problem_id} ({dim}D, Instance {instance_id})")
print(f"True Optimum: {problem.true_optimum:.4f}")
print(f"Configuration: mode={mode}, noise_std={noise_std}, iterations={iterations}, budget={budget}")

### Execute Single-Problem Evolution

In [ ]:
storage_manager = initialize_storage("sqlite", root_dir / "data" / "db.sqlite3")

problem_cfg = {
    "problem_id": problem.problem_id,
    "dim": problem.dim,
    "instance_id": problem.instance_id,
}

clean_fn = partial(
    LLaMEASession,
    problem=problem,
    llm=llm,
    noise_std=0.0,
    run_id=1,
    checkpoint_dir=root_dir / "data" / "checkpoints",
)
noisy_fn = partial(
    LLaMEASession,
    problem=problem,
    llm=llm,
    noise_std=0.5,
    run_id=1,
    checkpoint_dir=root_dir / "data" / "checkpoints",
)

jobs = [
    EvolutionJob("clean", clean_fn),
    EvolutionJob("noisy", noisy_fn),
]

results = dispatch(
    jobs=jobs,
    storage_manager=storage_manager,
    checkpoint_dir=root_dir / "data" / "checkpoints"
)

best_clean = results["clean"].best_solution
best_noisy = results["noisy"].best_solution

print(f"[OK] Clean evolution complete. Best Solution: {getattr(best_clean, 'name', 'N/A')} | Fitness: {getattr(best_clean, 'fitness', 'N/A')}")
print(f"[OK] Noisy evolution complete. Best Solution: {getattr(best_noisy, 'name', 'N/A')} | Fitness: {getattr(best_noisy, 'fitness', 'N/A')}")
print("[OK] Saved both runs and cleaned up checkpoint files.")

## 3. Section 2: Multi-Problem Batch Evolution Loop
Run LLaMEA evolution sequentially across multiple problem IDs. Results are persisted to the database after each problem run.

In [ ]:
# Batch Configuration
batch_problem_ids = [1, 2, 3, 4, 5]
dim = 3
noise_std = 0.05
iterations = 5
max_evaluations = 1000
mode = "noisy" if noise_std > 0.0 else "clean"

print(f"Starting batch evolution loop for problems: {batch_problem_ids} ({dim}D)")
print(f"Configuration: mode={mode}, noise_std={noise_std}, iterations={iterations}, max_evaluations={max_evaluations}")

In [ ]:
if llm is None:
    print("Error: Local LLM server must be running to execute batch evolution.")
else:
    # Setup storage
    db_url = os.environ.get("DATABASE_URL")
    if db_url:
        if db_url.startswith("sqlite:///"):
            db_path = Path(db_url[10:])
        else:
            db_path = Path(db_url)
        if not db_path.is_absolute():
            db_path = root_dir / db_path
    else:
        db_path = root_dir / "data" / "db.sqlite3"

    store = initialize_storage("sqlite", db_path)
    
    print(f"Starting batch orchestration loop for problems: {batch_problem_ids} ({dim}D)")
    
    for pid in batch_problem_ids:
        print(f"\n>>> Evolving algorithm for BBOB-{pid} concurrently...")
        problem_cfg = {
            "problem_id": pid,
            "dim": dim,
            "instance_id": 1,
        }
        try:
            clean_fn = partial(
                LLaMEASession,
                problem=BBOBProblem(problem_id=pid, dim=dim, instance_id=1),
                llm=llm,
                noise_std=0.0,
                run_id=1,
                checkpoint_dir=root_dir / "data" / "checkpoints",
            )
            noisy_fn = partial(
                LLaMEASession,
                problem=BBOBProblem(problem_id=pid, dim=dim, instance_id=1),
                llm=llm,
                noise_std=0.5,
                run_id=1,
                checkpoint_dir=root_dir / "data" / "checkpoints",
            )

            jobs = [
                EvolutionJob("clean", clean_fn),
                EvolutionJob("noisy", noisy_fn),
            ]

            dispatch(
                jobs=jobs,
                storage_manager=store,
                checkpoint_dir=root_dir / "data" / "checkpoints"
            )
            print(f"Saved concurrent runs for BBOB-{pid} to database.")
        except Exception as e:
            print(f"Failed to evolve algorithm for BBOB-{pid}: {e}")
            
    print("\nBatch evolution loop processing complete.")